In [21]:
from pathlib import Path
import ast

import pandas as pd

In [22]:
fp = Path("DST/extracted_entities.csv")


# fp = Path("data/extracted_entities_2026_06_03.csv")
df_input = pd.read_csv(fp)
df_input = df_input.rename(columns={"geographic location" : "location"})

#, index_col=0)
# Entity values to python objects
entity_cols = ['person', 'location', 'organization']
df_input[entity_cols] = df_input[entity_cols].map(ast.literal_eval)
df_input

,filename,paragraph,sentence,person,location,organization,historical_event
0,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...","Bij het bezoek, dat door den Koning dezer dage...","[{'start': 89, 'end': 94, 'text': 'Z. M.', 'la...",[],[],[]
1,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...","adjudant den majoor A. E. Mansveldt, Zr.","[{'start': 20, 'end': 35, 'text': 'A. E. Mansv...",[],[],[]
2,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...",Ms. stalmeester W. C. baron Snouckaert van Sch...,[],[],[],[]
3,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...","ordonnansofiicier, den kapitein H. J. M. C. ba...",[],[],[],[]
4,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...",H. M. de Koningin zal vergezeld worden door me...,"[{'start': 0, 'end': 17, 'text': 'H. M. de Kon...",[],[],[]
...,...,...,...,...,...,...,...
46460,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,"â Te .Muntok en Herawang viek vele regens, i...",[],"[{'start': 8, 'end': 14, 'text': 'Muntok', 'la...",[],"[{'start': 78, 'end': 83, 'text': 'wedei', 'la..."
46461,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,n de 52 op nltinie !â¢ â¢ â¢ ml.er handelin...,[],[],[],[]
46462,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,"i overhed I persoon, terwijl in het distrikt P...",[],[],[],"[{'start': 65, 'end': 76, 'text': 'Januarij 11..."
46463,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,"Dt oogst, waarmede zich de bevolking onledig h...",[],[],[],[]


In [23]:
# Optional: reduce irrelevant names by only keeping sentences that contain 'Nias'
# and the sentences before and after it (if they are from the same article)

df = df_input.copy()

# If you don't want to filter, comment out the following lines:

df['relevant_sentence'] = False
df['relevant_sentence'] = df.eval("sentence.str.contains('Nias')")
# Also assign sentence before and after to relevant_sentence (if it's from the same article (i.e. filename))
df['is_same_as_previous'] = df['filename'] == df['filename'].shift(1)
df['is_same_as_next'] = df['filename'] == df['filename'].shift(-1)

# Set relevant_sentence to True for sentences before and after relevant sentences
df['relevant_sentence'] = df['relevant_sentence'] | (df['is_same_as_previous'] & df['relevant_sentence'].shift(1, fill_value=False)) | (df['is_same_as_next'] & df['relevant_sentence'].shift(-1, fill_value=False))


df = df.query("relevant_sentence")
df

,filename,paragraph,sentence,person,location,organization,historical_event,relevant_sentence,is_same_as_previous,is_same_as_next
30,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...",Kaakebeen en 25 onder-officieren en manschappe...,[],"[{'start': 55, 'end': 70, 'text': 'Goenong Sit...",[],[],True,True,True
31,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...","civilen en militairen gezaghebber van Nias, de...",[],"[{'start': 38, 'end': 42, 'text': 'Nias', 'lab...",[],[],True,True,True
32,MMKB19_000142099_mpeg21_a00003.json,"Bij het bezoek, dat door den Koning dezer dage...","Meijer, 19 soldaten qn een mortier","[{'start': 0, 'end': 6, 'text': 'Meijer', 'lab...",[],[],[],True,True,True
149,MMKB19_000142109_mpeg21_a00028.json,"uv ci ^ BATAVIA, den 29â Maart 1863. Het alg...",Deze aardbeving was groot genoeg om van zich t...,[],"[{'start': 92, 'end': 98, 'text': 'Onrust', 'l...",[],"[{'start': 0, 'end': 15, 'text': 'Deze aardbev...",True,True,True
150,MMKB19_000142109_mpeg21_a00028.json,"uv ci ^ BATAVIA, den 29â Maart 1863. Het alg...",//Veel onaangenamer gebeurtenissen hebben er e...,[],"[{'start': 55, 'end': 62, 'text': 'Sumatra', '...",[],[],True,True,True
...,...,...,...,...,...,...,...,...,...,...
46445,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,"Itel /.. K. stc Maat â 'i Wind, die in de eer...",[],"[{'start': 72, 'end': 78, 'text': 'Padang', 'l...",[],[],True,True,True
46446,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,tevens de kontroleur van laatstgenoemd eiland ...,[],[],[],[],True,True,True
46452,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,"De vele regens waren, vooral in bel Sol (Padan...",[],"[{'start': 32, 'end': 39, 'text': 'bel Sol', '...",[],"[{'start': 0, 'end': 14, 'text': 'De vele rege...",True,True,True
46453,ddd_110534141_mpeg21_a0033.json,â (Februarij). Behoudens enkele gevallen i-a...,"Aan de piangans, welke in het N padigewas scha...",[],"[{'start': 85, 'end': 89, 'text': 'Nias', 'lab...",[],[],True,True,True


### Create dataframe with all persons

In [24]:
# Create a dataframe with only persons, one row per person
persons = df[['filename', 'person']]\
    .explode('person')\
    .dropna(subset=['person'])\
    .reset_index(drop=True)

persons_df = pd.concat([persons, pd.DataFrame(persons['person'].tolist())], axis=1).drop(columns=['person'])
print(f"Unieke personen: {persons_df.text.nunique()}")
persons_df

Unieke personen: 320


,filename,start,end,text,label,score
0,MMKB19_000142099_mpeg21_a00003.json,0,6,Meijer,persoon (naam),0.877176
1,MMKB19_000142109_mpeg21_a00028.json,112,128,Reinier Claeszen,persoon (naam),0.783604
2,MMKB19_000142109_mpeg21_a00028.json,346,355,Maleijers,persoon (naam),0.625696
3,MMKB19_000142123_mpeg21_a00020.json,96,108,heer Francis,persoon (naam),0.641545
4,MMKB19_000142123_mpeg21_a00020.json,188,204,heer van Kerchem,persoon (naam),0.743865
...,...,...,...,...,...,...
686,ddd_110532335_mpeg21_a0040.json,3,8,Julij,persoonsnaam,0.784169
687,ddd_110532841_mpeg21_a0043.json,0,11,Willemsorde,persoonsnaam,0.607359
688,ddd_110532849_mpeg21_a0033.json,6,22,W. S. J. Docters,persoonsnaam,0.611494
689,ddd_110533902_mpeg21_a0021.json,67,75,de Sahan,persoonsnaam,0.679414


## Convert to embeddings, build and search index

We're creating a 'FAISS' vector-index, to store the embeddings, and which we can use to quickly search and retrieve candidates (https://github.com/facebookresearch/faiss). Below is just a basic index, we could optimize it a bit if it turns out to be too slow

In [25]:
from sentence_transformers import SentenceTransformer
import faiss

# https://rapidfuzz.github.io/RapidFuzz/index.html
from rapidfuzz import fuzz, distance

In [9]:
# !git clone https://huggingface.co/emanjavacas/GysBERT

In [10]:
# import torch
# from sentence_transformers import SentenceTransformer

# embed_model = "GysBERT"

# if torch.cuda.is_available():
#     device = "cuda"
# elif torch.backends.mps.is_available():
#     device = "mps"
# else:
#     device = "cpu"

# # 1. Load a pretrained Sentence Transformer model
# model = SentenceTransformer(embed_model, device=device, trust_remote_code=True)

In [11]:
# !hf download emanjavacas/GysBERT

In [26]:
import torch

# _original_torch_load = torch.load

# def torch_load_compat(*args, **kwargs):
#     kwargs.setdefault("weights_only", False)
#     return _original_torch_load(*args, **kwargs)

# torch.load = torch_load_compat

from sentence_transformers import SentenceTransformer

embed_model = "emanjavacas/GysBERT"

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer(embed_model, device=device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 112945.40it/s]
[transformers] BertModel LOAD REPORT from: emanjavacas/GysBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
## Create embeddings for all names in the dataframe
# Get list of unique names
names = persons_df.text.unique().tolist()
print(f"Number of unique names: {len(names)}")

# Create embeddings for the unique names
embeddings = model.encode(names)


Number of unique names: 320


In [28]:
# Create index
index = faiss.IndexFlatL2(embeddings.shape[1])
print(index.is_trained)

# Add embeddings to the index
index.add(embeddings)

print(index.ntotal)

True
320


In [29]:
# Get k nearest neighbors for each name
k = 10

DISTANCES, CANDIDATE_INDICES = index.search(embeddings, k)

In [ ]:
# create a dataframe with names in first column, top 10 similar names in the next columns

suggested_names = pd.DataFrame({
    "name": names,
    "similar_names": [
        {
            names[j]: round(score, 2)
            for j in CANDIDATE_INDICES[i, :]
            if (score := fuzz.WRatio(names[i], names[j])) > 70
        }
        for i in range(len(names))
    ],
})

SyntaxError: expected 'else' after 'if' expression (2924569147.py, line 5)

In [31]:
suggested_names.head(20)

,name,similar_names
0,Meijer,"{'Meijer': 100.0, 'Chinees': 30.77, 'Spanjaard..."
1,Reinier Claeszen,"{'Reinier Claeszen': 100.0, 'Reinier Claesen':..."
2,Maleijers,"{'Maleijers': 100.0, 'maleijers': 88.89, 'Male..."
3,heer Francis,"{'heer Francis': 100.0, 'heer Sievers': 58.33,..."
4,heer van Kerchem,"{'heer van Kerchem': 100.0, 'heer Steinmetz': ..."
5,Hertog Benard,"{'Hertog Benard': 100.0, 'Hertog Bernari': 88...."
6,Radja van Assahan,"{'Radja van Assahan': 100.0, 'Sultan van Atjin..."
7,C. de Grijs,"{'C. de Grijs': 100.0, 'C. J. G. de Bruin': 85..."
8,A. QuikbÃ¶rner,"{'A. QuikbÃ¶rner': 100.0, 'J. Stebler': 41.67,..."
9,W. Smith,"{'W. Smith': 100.0, 'H. Walter': 35.29, 'F. W...."


### Inspect results

In [23]:
# Name idx (from list of unique names) to inspect
name_idx = 1


# Levenshtein weights for insertion, deletion, substitution
levenshtein_weights = (2, 2, 1)  

name = names[name_idx]

# Create a DataFrame to display the results (one row per name-candidate pair)
results = pd.DataFrame({
    "name": [name] * k,
    "candidate": [names[j] for j in CANDIDATE_INDICES[name_idx]],
    "l2_distance": DISTANCES[name_idx]
})

## Add columns with fuzzy matching scores
results['wratio'] = results['candidate'].apply(lambda x: fuzz.WRatio(name, x))
results['token_sort_ratio'] = results['candidate'].apply(lambda x: fuzz.token_sort_ratio(name, x))
results['lcs_ratio'] = results['candidate'].apply(lambda x: distance.LCSseq.normalized_distance(name.lower(), x.lower()))
results['levenshtein_ratio'] = results['candidate'].apply(lambda x: distance.Levenshtein.normalized_distance(name.lower(), x.lower(), weights=levenshtein_weights))
results['jarowinkler'] = results['candidate'].apply(lambda x: distance.JaroWinkler.normalized_distance(name.lower(), x.lower()))

# Display results rounded to 2 decimal places for readability
results.round(2)

,name,candidate,l2_distance,wratio,token_sort_ratio,lcs_ratio,levenshtein_ratio,jarowinkler
0,Reinier Claeszen,Reinier Claeszen,0.000000,100.00,100.00,0.00,0.00,0.00
1,Reinier Claeszen,majoor Fritzen,242.570007,33.33,33.33,0.69,0.72,0.45
2,Reinier Claeszen,H. J. Nieuwkerk,263.739990,24.52,25.81,0.75,1.00,0.52
3,Reinier Claeszen,B. Migchelsen,288.380005,41.38,27.59,0.62,0.74,0.42
4,Reinier Claeszen,C. de Grijs,313.720001,28.15,29.63,0.81,0.86,0.48
5,Reinier Claeszen,K. G. Fritzen,323.100006,27.59,27.59,0.69,0.79,0.50
6,Reinier Claeszen,J. S. Roels,325.459991,29.63,22.22,0.75,0.86,0.48
7,Reinier Claeszen,De maleijer,354.010010,42.22,44.44,0.69,0.81,0.38
8,Reinier Claeszen,Radja van Assahan,354.170013,30.30,30.30,0.65,0.78,0.48
9,Reinier Claeszen,W. Smith,354.399994,16.36,8.33,0.88,0.96,0.71


In [ ]:
# based on the embedding give me top 10 names that are similar to the name in the first column. Then for each of those names, give me the levenshtein distance, 


